In [204]:
import pandas as pd
import matplotlib.pyplot as plt
import os

output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

df.to_csv(f"{output_dir}/processed_data.csv", index=False)

stats = df.describe()
stats.to_csv(f"{output_dir}/summary_statistics.csv")

In [206]:
df = pd.read_csv("frailty_raw.csv")
print(df)


   Height_in  Weight_lb  Age_yr  Grip_kg Frailty
0       65.8        112      30       30       N
1       71.5        136      19       31       N
2       69.4        153      45       29       N
3       68.2        142      22       28       Y
4       67.8        144      29       24       Y
5       68.7        123      50       26       N
6       69.8        141      51       22       Y
7       70.1        136      23       20       Y
8       67.9        112      17       19       N
9       66.8        120      39       31       N


In [207]:
df["Height_m"] = df["Height_in"] * 0.0254
df["Weight_kg"] = df["Weight_lb"] * 0.45359237
df["BMI"] = (df["Weight_kg"] / (df["Height_m"] ** 2)).round(2)

def age_group(x):
    if x < 30:
        return "<30"
    elif x <= 45:
        return "30-45"
    elif x <= 60:
        return "46-60"
    return ">60"

df["AgeGroup"] = df["Age_yr"].apply(age_group)
df["Frailty_binary"] = df["Frailty"].map({"Y": 1, "N": 0}).astype("int8")

dummies = pd.get_dummies(df["AgeGroup"], prefix="AgeGroup")
df = pd.concat([df, dummies], axis=1)

print(df[["Height_in", "Height_m", "Weight_lb", "Weight_kg", "BMI", "Age_yr", "AgeGroup", "Frailty_binary"]].head())

   Height_in  Height_m  Weight_lb  Weight_kg    BMI  Age_yr AgeGroup  \
0       65.8   1.67132        112  50.802345  18.19      30    30-45   
1       71.5   1.81610        136  61.688562  18.70      19      <30   
2       69.4   1.76276        153  69.399633  22.33      45    30-45   
3       68.2   1.73228        142  64.410117  21.46      22      <30   
4       67.8   1.72212        144  65.317301  22.02      29      <30   

   Frailty_binary  
0               0  
1               0  
2               0  
3               1  
4               1  


In [208]:
df.to_csv("frailty_processed.csv", index=False)

print("\n Processed data saved as frailty_processed.csv")



 Processed data saved as frailty_processed.csv


In [213]:
summary = df.describe().T.loc[:, ["mean", "50%", "std"]]
summary = summary.rename(columns={"50%": "median"})

output_path = "Q1_Frailty_Workflow/reports"
os.makedirs(output_path, exist_ok=True)

with open(f"{output_path}/findings.md", "w") as report:
    report.write("# Frailty Dataset Report\n\n")
    report.write("## Summary Statistics\n\n")
    report.write(summary.to_markdown())
    report.write("\n")
print("Summary Table:")
print(summary)
print("\nCorrelation between Grip strength and Frailty_binary:", round(correlation, 3))
os.makedirs("findings",exist_ok=True)
summary.to_csv("findings/summary_statistics.csv")

Summary Table:
                      mean      median        std
Height_in        68.600000   68.450000   1.670662
Weight_lb       131.900000  136.000000  14.231811
Age_yr           32.500000   29.500000  12.860361
Grip_kg          26.000000   27.000000   4.521553
Height_m          1.742440    1.738630   0.042435
Weight_kg        59.828834   61.688562   6.455441
BMI              19.682000   19.185000   1.780972
Frailty_binary    0.400000    0.000000   0.516398

Correlation between Grip strength and Frailty_binary: -0.476
